# TranAD Telemetry Anomaly Detection Demo

This notebook demonstrates anomaly detection on simulated Prometheus-like telemetry data.

Pipeline:
1. Generate telemetry data (pods, events, CPU, memory)
2. Visualize metrics
3. Encode categorical features
4. Normalize features
5. Create sliding windows
6. Train a TranAD-style model
7. Evaluate anomaly detection performance
8. Visualize attention and anomaly scores
9. Simulate streaming detection


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

np.random.seed(42)
torch.manual_seed(42)

print("Libraries loaded")

## Step 1 — Simulate Prometheus-style telemetry

We generate synthetic telemetry representing pods performing operations.

Features:
- pod
- event
- eventType
- cpuUsage
- memoryUsage


In [ ]:
pods=["lcm-pod","monitor-pod","analytics-pod"]

events=[
"app_install",
"app_monitoring",
"app_uninstall",
"log_retrieval",
"log_update",
"push_image",
"pull_image"]

eventType_map={
"app_install":"peak",
"push_image":"peak",
"pull_image":"peak",
"log_retrieval":"peak",
"app_monitoring":"steady",
"app_uninstall":"steady",
"log_update":"steady"}

rows=[]
time_steps=3000

for t in range(time_steps):

    pod=np.random.choice(pods)
    event=np.random.choice(events)
    eventType=eventType_map[event]

    if pod=="lcm-pod":
        base_cpu=35
        base_mem=400
    elif pod=="monitor-pod":
        base_cpu=30
        base_mem=300
    else:
        base_cpu=45
        base_mem=450

    if eventType=="steady":
        cpu=np.random.normal(base_cpu,4)
        mem=np.random.normal(base_mem,30)
    else:
        cpu=np.random.normal(base_cpu+30,6)
        mem=np.random.normal(base_mem+400,60)

    rows.append([t,pod,event,eventType,cpu,mem,0])

df=pd.DataFrame(rows,columns=["time","pod","event","eventType","cpuUsage","memoryUsage","anomaly"])

# Inject anomalies
df.loc[1000:1020,"cpuUsage"]+=80
df.loc[1000:1020,"anomaly"]=1

df.loc[2000:2020,"memoryUsage"]+=600
df.loc[2000:2020,"anomaly"]=1

df.head()

## Step 2 — Visualize telemetry metrics

We plot CPU and memory usage over time.

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df['cpuUsage'])
plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU')
plt.show()

plt.figure(figsize=(12,4))
plt.plot(df['memoryUsage'])
plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory')
plt.show()

## Step 3 — Encode categorical features

Machine learning models require numerical inputs.

In [ ]:
enc_pod=LabelEncoder()
enc_event=LabelEncoder()
enc_type=LabelEncoder()

df['pod']=enc_pod.fit_transform(df['pod'])
df['event']=enc_event.fit_transform(df['event'])
df['eventType']=enc_type.fit_transform(df['eventType'])

df.head()

## Step 4 — Normalize features

Normalization ensures each feature contributes equally.

In [ ]:
features=["pod","event","eventType","cpuUsage","memoryUsage"]

scaler=StandardScaler()
data=scaler.fit_transform(df[features])

data[:5]

## Step 5 — Create sliding windows

Time-series models require sequences.

In [ ]:
window=20
X=[]
labels=[]

for i in range(len(data)-window):
    X.append(data[i:i+window])
    labels.append(df['anomaly'].iloc[i+window])

X=np.array(X)
labels=np.array(labels)

print("Dataset shape:",X.shape)

## Step 6 — TranAD-style model

This model uses attention and transformer encoding.

In [ ]:
class TranAD(nn.Module):
    def __init__(self,input_dim):
        super().__init__()

        self.attention=nn.MultiheadAttention(
            embed_dim=input_dim,
            num_heads=1,
            batch_first=True)

        self.encoder=nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=input_dim,
                nhead=1,
                dim_feedforward=64,
                batch_first=True),
            num_layers=2)

        self.decoder=nn.Linear(input_dim,input_dim)

    def forward(self,x):
        attn_out,weights=self.attention(x,x,x)
        encoded=self.encoder(attn_out)
        recon=self.decoder(encoded)
        return recon,weights

model=TranAD(X.shape[2])

criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

print(model)

## Step 7 — Train model

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
X,labels,test_size=0.3,shuffle=False)

X_train=torch.tensor(X_train,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)

epochs=15

for epoch in range(epochs):

    optimizer.zero_grad()

    output,_=model(X_train)

    loss=criterion(output,X_train)

    loss.backward()
    optimizer.step()

    print("Epoch",epoch,"Loss",loss.item())

## Step 8 — Compute anomaly score

In [ ]:
model.eval()

with torch.no_grad():

    recon_train,_=model(X_train)
    recon_test,_=model(X_test)

train_errors=torch.mean((X_train-recon_train)**2,dim=(1,2)).numpy()
test_errors=torch.mean((X_test-recon_test)**2,dim=(1,2)).numpy()

threshold=np.mean(train_errors)+3*np.std(train_errors)

pred=(test_errors>threshold).astype(int)

precision=precision_score(y_test,pred)
recall=recall_score(y_test,pred)
f1=f1_score(y_test,pred)

print("Precision",precision)
print("Recall",recall)
print("F1",f1)

## Step 9 — Attention heatmap

In [ ]:
with torch.no_grad():
    _,weights=model(X_test[:1])

plt.figure(figsize=(6,5))
sns.heatmap(weights[0].numpy())
plt.title("Attention Heatmap")
plt.show()

## Step 10 — Grafana-style anomaly dashboard

In [ ]:
fig,axs=plt.subplots(3,1,figsize=(12,10))

axs[0].plot(df['cpuUsage'])
axs[0].set_title("CPU Usage")

axs[1].plot(df['memoryUsage'])
axs[1].set_title("Memory Usage")

axs[2].plot(test_errors,label="Anomaly Score")
axs[2].axhline(threshold,color="red")

anom_idx=np.where(pred==1)[0]

axs[2].scatter(anom_idx,test_errors[pred==1],color="red")

axs[2].legend()

plt.show()